In [1]:
# %pip install -q openai numpy python-dotenv

In [2]:
# %pip install -q -U openai

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

client = OpenAI()

Key loaded: True


In [5]:
# response = client.chat.completions.create(
#     model="gpt-6.1-sol",
#     messages=[
#         {"role": "user", "content": "Say hello and explain RAG in one sentence."}
#     ],
# )
# print(response.choices[0].message.content)

MODEL = "gpt-6.1-sol"

def ask(question, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
    )
    return response.choices[0].message.content

In [6]:
document = """
Nour Cafe - Internal Policies

Opening hours: Sunday to Thursday 8:00 AM to 11:00 PM. Closed on Fridays.
Refund policy: Customers can return unopened coffee bags within 14 days.
Opened bags cannot be returned.
Loyalty program: Every 10th coffee is free. Points expire after 90 days.
Manager: Layla Haddad. Support email: help@nourcafe.example
"""

In [7]:
print(ask("What is the refund policy at Nour Cafe, and who is the manager?"))

Which Nour Cafe do you mean? Please share its city, address, or website so I can identify the right café and check its refund policy and manager’s name.


In [8]:
system_prompt = f"""Answer the question using ONLY the document below.
If the answer is not in the document, say you don't know.

<document>
{document}
</document>"""

print(ask("What is the refund policy at Nour Cafe, and who is the manager?", system=system_prompt))
print("-----")
print(ask("What is the capital of France?", system=system_prompt))

Nour Cafe accepts returns of unopened coffee bags within 14 days. Opened bags cannot be returned.

The manager is Layla Haddad.
-----
I don't know.


In [9]:
big_doc = document * 2000
print(len(big_doc), "characters ≈", len(big_doc) // 4, "tokens")

688000 characters ≈ 172000 tokens


Chunking

In [11]:
def chunk_text(text, chunk_size=150, overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = end - overlap
    return chunks

chunks = chunk_text(document)
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c)

--- chunk 0 (149 chars) ---
Nour Cafe - Internal Policies

Opening hours: Sunday to Thursday 8:00 AM to 11:00 PM. Closed on Fridays.
Refund policy: Customers can return unopened
--- chunk 1 (149 chars) ---
Customers can return unopened coffee bags within 14 days.
Opened bags cannot be returned.
Loyalty program: Every 10th coffee is free. Points expire a
--- chunk 2 (103 chars) ---
offee is free. Points expire after 90 days.
Manager: Layla Haddad. Support email: help@nourcafe.example
